# Lucent Weight Analysis for Machine Unlearning

This notebook analyzes the most influential weights, neurons, layers, and channels affected by machine unlearning using Lucent visualization tools.

**Perfect for Google Colab!** 🚀

## Features:
- 🔍 Load your weight analysis results
- 🎯 Identify top affected components
- 🎨 Generate Lucent visualizations
- 📊 Interactive analysis of unlearning effects

## 1. Install and Import Lucent

First, let's install Lucent and set up the environment for Colab.

In [ ]:
# Install Lucent in Colab
!pip install lucent

# Standard imports
import torch
import torch.nn as nn
import torchvision.models as models
import numpy as np
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, OrderedDict

# Lucent imports
from lucent.optvis import render, objectives, transforms
from lucent.modelzoo import util

# Set up plotting
plt.style.use('default')
sns.set_palette("husl")

print("✅ Lucent and dependencies installed successfully!")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🎯 Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

## 2. Load Weight Analysis Results

Load your pre-computed weight analysis results to identify the most affected components.

In [ ]:
# FOR GOOGLE COLAB: Load Multi-Experiment JSON Results
from google.colab import drive, files
import json
import os

print("🔗 MULTI-EXPERIMENT ANALYSIS LOADER")
print("="*50)

# Mount Google Drive
try:
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully!")
    
    # Define paths for all three experiments
    experiment_paths = {
        '10percent': '/content/drive/MyDrive/10percent_lucent_targets_real.json',
        '20percent': '/content/drive/MyDrive/20percent_lucent_targets_real.json',
        '30percent': '/content/drive/MyDrive/30percent_lucent_targets_real.json',
        'comparison': '/content/drive/MyDrive/experiment_comparison.json',
        'unified': '/content/drive/MyDrive/unified_lucent_targets.json'
    }
    
    # Load all available experiments
    experiments_data = {}
    
    for exp_name, path in experiment_paths.items():
        if os.path.exists(path):
            print(f"🎯 Found {exp_name}: {path}")
            with open(path, 'r') as f:
                experiments_data[exp_name] = json.load(f)
            print(f"✅ Loaded {exp_name} with {len(experiments_data[exp_name])} targets/data")
        else:
            print(f"⚠️  {exp_name} not found: {path}")
    
    if experiments_data:
        print(f"\n🔬 Successfully loaded {len(experiments_data)} experiment datasets!")
        for exp_name in experiments_data.keys():
            print(f"   - {exp_name}")
        use_drive = True
    else:
        print("❌ No experiment files found in Google Drive")
        use_drive = False
        
except Exception as e:
    print(f"❌ Drive mount failed: {e}")
    use_drive = False

# Fallback to manual upload if Drive doesn't work
if not use_drive:
    print("\n📤 Manual Upload Mode")
    print("Upload your experiment JSON files one by one:")
    
    experiments_data = {}
    
    for exp_name in ['10percent', '20percent', '30percent']:
        print(f"\n📤 Upload {exp_name}_lucent_targets_real.json:")
        try:
            uploaded = files.upload()
            if uploaded:
                file_name = list(uploaded.keys())[0]
                with open(file_name, 'r') as f:
                    experiments_data[exp_name] = json.load(f)
                print(f"✅ Loaded {exp_name} experiment")
        except:
            print(f"⚠️  Skipped {exp_name}")

print(f"\n🎯 EXPERIMENT SELECTOR")
print("-"*30)

# Create experiment selector function
def select_experiment(experiment_name):
    """Select which experiment to analyze"""
    if experiment_name in experiments_data:
        selected_data = experiments_data[experiment_name]
        print(f"🎯 Selected: {experiment_name}")
        print(f"📊 Available targets: {len(selected_data)}")
        return selected_data
    else:
        print(f"❌ Experiment {experiment_name} not available")
        print(f"Available: {list(experiments_data.keys())}")
        return None

# Set default experiment (you can change this)
current_experiment = "10percent"  # Change to "20percent" or "30percent" as needed
selected_data = select_experiment(current_experiment)

if selected_data:
    print(f"\n✅ Ready to analyze {current_experiment} forgetting experiment!")
    print(f"💡 To switch experiments, call: select_experiment('20percent')")
else:
    print("\n❌ No valid experiment data loaded")

print(f"\n📋 AVAILABLE EXPERIMENTS:")
for exp_name in experiments_data.keys():
    if exp_name not in ['comparison', 'unified']:
        data_size = len(experiments_data[exp_name]) if experiments_data[exp_name] else 0
        print(f"   • {exp_name}: {data_size} targets")

## 2.1 Multi-Experiment Analysis & Comparison

Compare results across different forgetting ratios and extract insights.

In [ ]:
# Compare results across experiments
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def compare_experiments(results_dict):
    """Compare analysis results across different forgetting ratios."""
    experiments = list(results_dict.keys())
    
    print(f"🔍 Comparing {len(experiments)} experiments: {experiments}")
    
    # 1. Compare most influenced layers
    print("\n" + "="*50)
    print("MOST INFLUENCED LAYERS COMPARISON")
    print("="*50)
    
    layer_comparison = {}
    for exp_name, results in results_dict.items():
        top_layers = results['layer_analysis']['top_influenced_layers'][:5]
        layer_comparison[exp_name] = [layer['layer'] for layer in top_layers]
        
        print(f"\n{exp_name.upper()}:")
        for i, layer in enumerate(top_layers, 1):
            print(f"  {i}. {layer['layer']}: {layer['avg_change']:.6f}")
    
    # 2. Compare channel analysis
    print("\n" + "="*50)
    print("CHANNEL ANALYSIS COMPARISON")
    print("="*50)
    
    for exp_name, results in results_dict.items():
        channel_stats = results['channel_analysis']['summary']
        print(f"\n{exp_name.upper()}:")
        print(f"  Total channels: {channel_stats['total_channels']}")
        print(f"  High influence channels (>0.001): {channel_stats['high_influence_channels']}")
        print(f"  Very high influence channels (>0.01): {channel_stats['very_high_influence_channels']}")
        print(f"  Max channel influence: {channel_stats['max_channel_influence']:.6f}")
    
    # 3. Compare weight statistics
    print("\n" + "="*50)
    print("WEIGHT ANALYSIS COMPARISON")
    print("="*50)
    
    for exp_name, results in results_dict.items():
        weight_stats = results['weight_analysis']['summary']
        print(f"\n{exp_name.upper()}:")
        print(f"  Total weights: {weight_stats['total_weights']:,}")
        print(f"  Changed weights (>1e-6): {weight_stats['changed_weights']:,}")
        print(f"  Significantly changed (>0.001): {weight_stats['significantly_changed']:,}")
        print(f"  Max weight change: {weight_stats['max_change']:.6f}")
        print(f"  Average absolute change: {weight_stats['avg_abs_change']:.8f}")
    
    return layer_comparison

# Run comparison if multiple experiments loaded
if len(experiment_results) > 1:
    layer_comparison = compare_experiments(experiment_results)
    
    # Create visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Multi-Experiment Comparison', fontsize=16)
    
    # Plot 1: High influence channels comparison
    exp_names = list(experiment_results.keys())
    high_influence_counts = []
    very_high_influence_counts = []
    
    for exp_name in exp_names:
        stats = experiment_results[exp_name]['channel_analysis']['summary']
        high_influence_counts.append(stats['high_influence_channels'])
        very_high_influence_counts.append(stats['very_high_influence_channels'])
    
    x = range(len(exp_names))
    axes[0,0].bar([i-0.2 for i in x], high_influence_counts, 0.4, label='High influence (>0.001)', alpha=0.7)
    axes[0,0].bar([i+0.2 for i in x], very_high_influence_counts, 0.4, label='Very high influence (>0.01)', alpha=0.7)
    axes[0,0].set_xlabel('Experiment')
    axes[0,0].set_ylabel('Number of Channels')
    axes[0,0].set_title('High Influence Channels by Experiment')
    axes[0,0].set_xticks(x)
    axes[0,0].set_xticklabels(exp_names, rotation=45)
    axes[0,0].legend()
    
    # Plot 2: Weight change statistics
    max_changes = []
    avg_changes = []
    
    for exp_name in exp_names:
        stats = experiment_results[exp_name]['weight_analysis']['summary']
        max_changes.append(stats['max_change'])
        avg_changes.append(stats['avg_abs_change'])
    
    axes[0,1].bar(x, max_changes, alpha=0.7)
    axes[0,1].set_xlabel('Experiment')
    axes[0,1].set_ylabel('Max Weight Change')
    axes[0,1].set_title('Maximum Weight Changes')
    axes[0,1].set_xticks(x)
    axes[0,1].set_xticklabels(exp_names, rotation=45)
    
    # Plot 3: Top layer influences
    layer_data = []
    for exp_name in exp_names:
        top_layers = experiment_results[exp_name]['layer_analysis']['top_influenced_layers'][:3]
        for layer in top_layers:
            layer_data.append({
                'experiment': exp_name,
                'layer': layer['layer'],
                'influence': layer['avg_change']
            })
    
    df_layers = pd.DataFrame(layer_data)
    if not df_layers.empty:
        layer_pivot = df_layers.pivot(index='layer', columns='experiment', values='influence')
        sns.heatmap(layer_pivot, annot=True, fmt='.6f', ax=axes[1,0], cmap='YlOrRd')
        axes[1,0].set_title('Top Layer Influences Heatmap')
    
    # Plot 4: Changed weights percentage
    changed_percentages = []
    for exp_name in exp_names:
        stats = experiment_results[exp_name]['weight_analysis']['summary']
        percentage = (stats['changed_weights'] / stats['total_weights']) * 100
        changed_percentages.append(percentage)
    
    axes[1,1].bar(x, changed_percentages, alpha=0.7, color='coral')
    axes[1,1].set_xlabel('Experiment')
    axes[1,1].set_ylabel('Percentage of Changed Weights')
    axes[1,1].set_title('Percentage of Weights Changed (>1e-6)')
    axes[1,1].set_xticks(x)
    axes[1,1].set_xticklabels(exp_names, rotation=45)
    
    plt.tight_layout()
    plt.show()
    
else:
    print("Load multiple experiments to enable comparison analysis.")
    if len(experiment_results) == 1:
        exp_name = list(experiment_results.keys())[0]
        print(f"Currently loaded: {exp_name}")
        print("Upload additional JSON files to compare experiments.")

## 3. Lucent Feature Visualization Integration

Use the analysis results to target specific layers and channels for Lucent visualization.

In [ ]:
# Install Lucent (run once)
!pip install lucent

import torch
import torch.nn as nn
import torchvision.models as models
from lucent.optvis import render, param, transform, objectives
from lucent.modelzoo import inceptionv1
import numpy as np
import matplotlib.pyplot as plt

def load_resnet50_model(model_path):
    """Load the ResNet50 model for visualization."""
    model = models.resnet50(pretrained=False)
    model.fc = nn.Linear(model.fc.in_features, 200)  # Tiny ImageNet has 200 classes
    
    # Load the model weights
    checkpoint = torch.load(model_path, map_location='cpu')
    if 'state_dict' in checkpoint:
        model.load_state_dict(checkpoint['state_dict'])
    else:
        model.load_state_dict(checkpoint)
    
    model.eval()
    return model

def create_lucent_targets(experiment_results, experiment_name, top_n=5):
    """Create Lucent visualization targets from analysis results."""
    if experiment_name not in experiment_results:
        print(f"Experiment {experiment_name} not found!")
        return []
    
    results = experiment_results[experiment_name]
    targets = []
    
    # Get top influenced channels for visualization
    if 'lucent_targets' in results:
        lucent_data = results['lucent_targets']
        
        print(f"🎯 Creating visualization targets for {experiment_name}:")
        print("="*50)
        
        for i, target in enumerate(lucent_data[:top_n]):
            layer_name = target['layer']
            channel_idx = target['channel']
            influence = target['influence']
            
            print(f"{i+1}. Layer: {layer_name}, Channel: {channel_idx}, Influence: {influence:.6f}")
            
            # Create objective for this channel
            # Note: You'll need to adapt this based on your model's layer structure
            targets.append({
                'name': f"{layer_name}_ch{channel_idx}",
                'layer': layer_name,
                'channel': channel_idx,
                'influence': influence,
                'objective': f"objectives.channel('{layer_name}', {channel_idx})"
            })
    
    return targets

# Create visualization targets for selected experiment
def visualize_influenced_channels(model, targets, device='cpu'):
    """Visualize the most influenced channels using Lucent."""
    model = model.to(device)
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for i, target in enumerate(targets[:6]):  # Visualize top 6
        try:
            # Create the channel objective
            layer_name = target['layer']
            channel_idx = target['channel']
            
            # Render the feature visualization
            # Note: This is a simplified example - you may need to adapt layer names
            # based on your actual model structure
            
            print(f"Visualizing {target['name']}...")
            
            # For ResNet50, layer names might need mapping
            # Example: layer4.2.conv3 -> model.layer4[2].conv3
            objective = objectives.channel(layer_name, channel_idx)
            
            # Render visualization
            img = render.render_vis(
                model, 
                objective,
                param_f=lambda: param.image(128),
                optimizer=torch.optim.Adam,
                transforms=[transform.pad(2), transform.jitter(8), transform.random_scale([1, 0.975, 1.025, 0.95, 1.05])],
                thresholds=[256],
                show_inline=False,
                save_image=False
            )
            
            if img is not None and len(img) > 0:
                axes[i].imshow(img[0])
                axes[i].set_title(f"{target['name']}\nInfluence: {target['influence']:.6f}")
                axes[i].axis('off')
            else:
                axes[i].text(0.5, 0.5, f"Failed to render\n{target['name']}", 
                           ha='center', va='center', transform=axes[i].transAxes)
                axes[i].axis('off')
                
        except Exception as e:
            print(f"Error visualizing {target['name']}: {e}")
            axes[i].text(0.5, 0.5, f"Error rendering\n{target['name']}", 
                        ha='center', va='center', transform=axes[i].transAxes)
            axes[i].axis('off')
    
    # Hide unused subplots
    for j in range(len(targets), len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    plt.suptitle(f'Most Influenced Channels Visualization', y=1.02, fontsize=16)
    plt.show()

print("🎨 Lucent visualization functions ready!")
print("Next steps:")
print("1. Specify your model path")
print("2. Select experiment for visualization")
print("3. Run visualization")

In [ ]:
# CONFIGURE YOUR PATHS AND RUN VISUALIZATION
# =============================================

# 1. Set your model paths (update these paths)
baseline_model_path = "/path/to/your/baseline_model.pth"  # Update this path
unlearned_model_path = "/path/to/your/unlearned_model.pth"  # Update this path

# 2. Select experiment to visualize
experiment_to_visualize = "10_percent_forgetting"  # Change this to your desired experiment

# 3. Load model and create targets
try:
    print(f"Loading model for visualization...")
    model = load_resnet50_model(baseline_model_path)
    print("✅ Model loaded successfully!")
    
    # Create visualization targets
    if experiment_to_visualize in experiment_results:
        targets = create_lucent_targets(experiment_results, experiment_to_visualize, top_n=6)
        
        if targets:
            print(f"\n🎯 Created {len(targets)} visualization targets")
            
            # Run visualization
            print("🎨 Starting feature visualization...")
            visualize_influenced_channels(model, targets)
        else:
            print("❌ No targets created. Check if 'lucent_targets' exists in your JSON.")
    else:
        print(f"❌ Experiment '{experiment_to_visualize}' not found.")
        print(f"Available experiments: {list(experiment_results.keys())}")
        
except Exception as e:
    print(f"❌ Error: {e}")
    print("\n💡 Troubleshooting:")
    print("1. Make sure model paths are correct")
    print("2. Verify the experiment name exists")
    print("3. Check that JSON contains 'lucent_targets' section")

## 3. Analyze Most Influential Weights

Extract and rank the most influential weights across all experiments.

In [ ]:
def analyze_most_influential_weights(weight_analysis, top_k=20):
    """Extract the most influential weights across all experiments"""
    
    all_weights = []
    
    for exp_name, exp_data in weight_analysis.items():
        if 'layer_sensitivity' in exp_data:
            layer_data = exp_data['layer_sensitivity']
            
            for layer_name, layer_stats in layer_data.items():
                if isinstance(layer_stats, dict):
                    all_weights.append({
                        'experiment': exp_name,
                        'layer': layer_name,
                        'mean_change': layer_stats.get('mean_absolute_change', 0),
                        'max_change': layer_stats.get('max_absolute_change', 0),
                        'relative_change': layer_stats.get('mean_relative_change', 0),
                        'percentage_changed': layer_stats.get('percentage_changed', 0),
                        'layer_type': categorize_layer_type(layer_name)
                    })
    
    # Convert to DataFrame for analysis
    df = pd.DataFrame(all_weights)
    
    # Sort by relative change (most informative metric)
    df_sorted = df.sort_values('relative_change', ascending=False)
    
    return df_sorted.head(top_k)

def categorize_layer_type(layer_name):
    """Categorize layer by type for better analysis"""
    if 'conv' in layer_name and 'weight' in layer_name:
        return 'Conv Weight'
    elif 'bn' in layer_name and 'weight' in layer_name:
        return 'BN Weight'
    elif 'bn' in layer_name and 'bias' in layer_name:
        return 'BN Bias'
    elif 'bn' in layer_name and 'running_mean' in layer_name:
        return 'BN Running Mean'
    elif 'bn' in layer_name and 'running_var' in layer_name:
        return 'BN Running Var'
    elif 'fc' in layer_name or 'classifier' in layer_name:
        return 'FC Layer'
    else:
        return 'Other'

# Analyze most influential weights
print("🔍 MOST INFLUENTIAL WEIGHTS ANALYSIS")
print("="*50)

top_weights = analyze_most_influential_weights(weight_analysis, top_k=20)

print(f"📊 Top {len(top_weights)} Most Affected Weights:")
print("-" * 70)

for i, row in top_weights.iterrows():
    exp_short = row['experiment'].replace('random_forgetting_', '').replace('_RL_tweak_conservative', '')
    layer_short = row['layer'].replace('.weight', '').replace('.bias', '')
    
    print(f"{i+1:2d}. {layer_short:<25} | {exp_short:<8} | {row['layer_type']:<15}")
    print(f"    📈 Relative change: {row['relative_change']:.4f}")
    print(f"    🎯 Percentage changed: {row['percentage_changed']:.1f}%")
    print()

# Create visualization
plt.figure(figsize=(15, 8))
plt.subplot(1, 2, 1)
sns.barplot(data=top_weights.head(10), y='layer', x='relative_change', hue='layer_type')
plt.title('Top 10 Weights by Relative Change')
plt.xlabel('Relative Change')

plt.subplot(1, 2, 2)
layer_type_summary = top_weights.groupby('layer_type')['relative_change'].mean().sort_values(ascending=False)
plt.pie(layer_type_summary.values, labels=layer_type_summary.index, autopct='%1.1f%%')
plt.title('Distribution by Layer Type')

plt.tight_layout()
plt.show()

print(f"💡 Key Finding: {top_weights.iloc[0]['layer_type']} layers are most affected!")

## 4. Identify Important Neurons and Channels

Extract specific neurons and channels for Lucent visualization targeting.

In [ ]:
def generate_lucent_targets(weight_analysis, top_k=15):
    """Generate Lucent-compatible targeting strings for visualization"""
    
    lucent_targets = []
    
    for exp_name, exp_data in weight_analysis.items():
        if 'layer_sensitivity' in exp_data:
            layer_data = exp_data['layer_sensitivity']
            
            # Sort layers by relative change
            sorted_layers = sorted(layer_data.items(), 
                                 key=lambda x: x[1].get('mean_relative_change', 0) if isinstance(x[1], dict) else 0, 
                                 reverse=True)
            
            for layer_name, layer_stats in sorted_layers[:top_k]:
                if isinstance(layer_stats, dict) and 'conv' in layer_name and 'weight' in layer_name:
                    
                    # Convert layer name to Lucent target format
                    # Example: "layer1.0.conv1.weight" -> "layer1.0.conv1"
                    lucent_layer = layer_name.replace('.weight', '')
                    
                    # Extract block and layer info for channel targeting
                    if 'layer' in lucent_layer:
                        # For specific channel targeting, we'll add channel indices
                        # Based on typical ResNet structure
                        if 'layer1' in lucent_layer:
                            channels = [0, 16, 32, 48]  # Sample channels
                        elif 'layer2' in lucent_layer:
                            channels = [0, 32, 64, 96]
                        elif 'layer3' in lucent_layer:
                            channels = [0, 64, 128, 192]  
                        elif 'layer4' in lucent_layer:
                            channels = [0, 128, 256, 384]
                        else:
                            channels = [0, 10, 20, 30]
                        
                        for channel in channels[:2]:  # Top 2 channels per layer
                            lucent_targets.append({
                                'experiment': exp_name,
                                'target': f"{lucent_layer}:{channel}",
                                'layer': lucent_layer,
                                'channel': channel,
                                'change_magnitude': layer_stats.get('mean_relative_change', 0),
                                'type': 'channel'
                            })
                    
                    # Also add layer-level targets (without specific channels)
                    lucent_targets.append({
                        'experiment': exp_name,
                        'target': lucent_layer,
                        'layer': lucent_layer,
                        'channel': 'all',
                        'change_magnitude': layer_stats.get('mean_relative_change', 0),
                        'type': 'layer'
                    })
    
    # Sort by change magnitude and return top targets
    lucent_targets.sort(key=lambda x: x['change_magnitude'], reverse=True)
    return lucent_targets[:top_k]

# Generate Lucent targets
print("🎯 LUCENT VISUALIZATION TARGETS")
print("="*50)

lucent_targets = generate_lucent_targets(weight_analysis, top_k=15)

print(f"📌 Top {len(lucent_targets)} Lucent Targets for Visualization:")
print("-" * 70)

for i, target_info in enumerate(lucent_targets):
    exp_short = target_info['experiment'].replace('random_forgetting_', '').replace('_RL_tweak_conservative', '')
    
    print(f"{i+1:2d}. {target_info['target']:<30} | {target_info['type']:<8} | {exp_short}")
    print(f"    📊 Change magnitude: {target_info['change_magnitude']:.4f}")
    print()

# Create a summary DataFrame
targets_df = pd.DataFrame(lucent_targets)

# Visualize target distribution
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
type_counts = targets_df['type'].value_counts()
plt.pie(type_counts.values, labels=type_counts.index, autopct='%1.1f%%')
plt.title('Target Types Distribution')

plt.subplot(1, 3, 2)
exp_counts = targets_df['experiment'].value_counts()
plt.bar(range(len(exp_counts)), exp_counts.values)
plt.xticks(range(len(exp_counts)), [exp.replace('random_forgetting_', '').replace('_RL_tweak_conservative', '') for exp in exp_counts.index], rotation=45)
plt.title('Targets per Experiment')
plt.ylabel('Number of Targets')

plt.subplot(1, 3, 3)
plt.hist(targets_df['change_magnitude'], bins=10, alpha=0.7)
plt.title('Change Magnitude Distribution')
plt.xlabel('Change Magnitude')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

print("\n💡 These targets are ready for Lucent visualization!")

## 5. Load Model and Setup Lucent Visualization

Load your trained model and set up for Lucent visualization.

In [ ]:
# FOR GOOGLE COLAB: Load your model or use pretrained
from google.colab import files
import torch
import torch.nn as nn
import torchvision.models as models

# Option 1: Upload your model (if you have it)
print("📤 Upload your model file (optional - RLcheckpoint.pth.tar):")
print("   Skip this if you want to use a pretrained model for demonstration")

try:
    model_files = files.upload()
    
    def load_unlearn_model(model_path, num_classes=200):
        """Load your unlearned model for visualization"""
        
        # Create ResNet50 for Tiny ImageNet (200 classes)
        model = models.resnet50(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        
        # Load checkpoint
        checkpoint = torch.load(model_path, map_location='cpu')
        if 'state_dict' in checkpoint:
            model.load_state_dict(checkpoint['state_dict'], strict=False)
        else:
            model.load_state_dict(checkpoint, strict=False)
        
        model.eval()
        return model
    
    # Load your model
    if model_files:
        model_path = list(model_files.keys())[0]
        model = load_unlearn_model(model_path)
        print(f"✅ Loaded your unlearned model: {model_path}")
    else:
        raise FileNotFoundError("No model uploaded")
        
except:
    # Option 2: Use pretrained model for demonstration
    print("🎯 Using pretrained ResNet50 for demonstration")
    print("   (This will still show you how the analysis works!)")
    model = models.resnet50(weights='IMAGENET1K_V1')
    model.eval()

print(f"🏗️  Model architecture: {model.__class__.__name__}")
print(f"📊 Total parameters: {sum(p.numel() for p in model.parameters()):,}")

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"🚀 Model loaded on: {device}")

# Print model structure for reference
print("\n🔍 Model layer structure (for targeting):")
for name, module in model.named_modules():
    if len(list(module.children())) == 0:  # Leaf modules only
        print(f"   {name}: {module.__class__.__name__}")
        if len(name.split('.')) <= 3:  # Don't show too deep
            continue

## 6. Visualize Most Influential Components

Now let's create Lucent visualizations for the most influential components identified from your weight analysis!

In [ ]:
def visualize_top_influenced_channels(model, lucent_targets, num_visualizations=6):
    """Visualize the most influenced channels using Lucent"""
    
    print(f"🎨 Generating visualizations for top {num_visualizations} most influenced components...")
    print("-" * 70)
    
    # Select top targets for visualization
    targets_to_visualize = lucent_targets[:num_visualizations]
    
    # Create a figure to display results
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for i, target_info in enumerate(targets_to_visualize):
        target = target_info['target']
        change_mag = target_info['change_magnitude']
        
        print(f"🎯 {i+1}. Visualizing: {target}")
        print(f"   📊 Change magnitude: {change_mag:.4f}")
        print(f"   🧪 From experiment: {target_info['experiment']}")
        
        try:
            # Generate visualization using Lucent
            # Using multiple optimization steps for better quality
            img = render.render_vis(
                model, 
                target,
                show_inline=False,
                thresholds=(256, 512),  # Multiple resolution steps
                show_image=False
            )
            
            # Display in subplot
            if hasattr(img, 'cpu'):
                img_np = img.cpu().numpy().transpose(1, 2, 0)
            else:
                img_np = np.array(img)
            
            axes[i].imshow(img_np)
            axes[i].set_title(f"{target}\\nChange: {change_mag:.4f}", fontsize=10)
            axes[i].axis('off')
            
            print(f"   ✅ Visualization complete!")
            
        except Exception as e:
            print(f"   ❌ Failed: {e}")
            # Create placeholder
            axes[i].text(0.5, 0.5, f"Failed to visualize\\n{target}", 
                        ha='center', va='center', transform=axes[i].transAxes)
            axes[i].set_title(f"{target}\\n(Failed)", fontsize=10)
            axes[i].axis('off')
        
        print()
    
    plt.suptitle('🎨 Lucent Visualizations of Most Influenced Components', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    return targets_to_visualize

# Run the visualization
visualized_targets = visualize_top_influenced_channels(model, lucent_targets, num_visualizations=6)

print("🎉 Visualization complete!")
print("💡 These images show what patterns the most affected neurons/channels respond to.")

## 7. Interactive Analysis Dashboard

Create an interactive analysis to explore different aspects of your unlearning results.

In [ ]:
def create_interactive_analysis_dashboard():
    """Create an interactive dashboard for exploring unlearning effects"""
    
    print("📊 INTERACTIVE ANALYSIS DASHBOARD")
    print("="*50)
    
    # 1. Experiment Comparison
    exp_comparison = {}
    for exp_name, exp_data in weight_analysis.items():
        if 'layer_sensitivity' in exp_data:
            layer_data = exp_data['layer_sensitivity']
            
            # Calculate summary statistics
            changes = [stats.get('mean_relative_change', 0) 
                      for stats in layer_data.values() 
                      if isinstance(stats, dict)]
            
            exp_comparison[exp_name] = {
                'avg_change': np.mean(changes) if changes else 0,
                'max_change': np.max(changes) if changes else 0,
                'affected_layers': len([c for c in changes if c > 0.1]),
                'total_layers': len(changes)
            }
    
    # Create comparison visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Plot 1: Average change by experiment
    exp_names = list(exp_comparison.keys())
    avg_changes = [exp_comparison[exp]['avg_change'] for exp in exp_names]
    
    axes[0,0].bar(range(len(exp_names)), avg_changes)
    axes[0,0].set_title('Average Weight Change by Experiment')
    axes[0,0].set_ylabel('Average Relative Change')
    axes[0,0].set_xticks(range(len(exp_names)))
    axes[0,0].set_xticklabels([exp.replace('random_forgetting_', '').replace('_RL_tweak_conservative', '') 
                               for exp in exp_names], rotation=45)
    
    # Plot 2: Max change by experiment
    max_changes = [exp_comparison[exp]['max_change'] for exp in exp_names]
    axes[0,1].bar(range(len(exp_names)), max_changes, color='orange')
    axes[0,1].set_title('Maximum Weight Change by Experiment')
    axes[0,1].set_ylabel('Maximum Relative Change')
    axes[0,1].set_xticks(range(len(exp_names)))
    axes[0,1].set_xticklabels([exp.replace('random_forgetting_', '').replace('_RL_tweak_conservative', '') 
                               for exp in exp_names], rotation=45)
    
    # Plot 3: Number of affected layers
    affected_layers = [exp_comparison[exp]['affected_layers'] for exp in exp_names]
    axes[1,0].bar(range(len(exp_names)), affected_layers, color='green')
    axes[1,0].set_title('Number of Highly Affected Layers')
    axes[1,0].set_ylabel('Layers with >0.1 Change')
    axes[1,0].set_xticks(range(len(exp_names)))
    axes[1,0].set_xticklabels([exp.replace('random_forgetting_', '').replace('_RL_tweak_conservative', '') 
                               for exp in exp_names], rotation=45)
    
    # Plot 4: Layer type analysis across all experiments
    layer_type_changes = defaultdict(list)
    for exp_name, exp_data in weight_analysis.items():
        if 'layer_sensitivity' in exp_data:
            layer_data = exp_data['layer_sensitivity']
            for layer_name, layer_stats in layer_data.items():
                if isinstance(layer_stats, dict):
                    layer_type = categorize_layer_type(layer_name)
                    change = layer_stats.get('mean_relative_change', 0)
                    layer_type_changes[layer_type].append(change)
    
    # Box plot of changes by layer type
    layer_types = list(layer_type_changes.keys())
    changes_by_type = [layer_type_changes[lt] for lt in layer_types]
    
    axes[1,1].boxplot(changes_by_type, labels=layer_types)
    axes[1,1].set_title('Weight Changes by Layer Type')
    axes[1,1].set_ylabel('Relative Change')
    axes[1,1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # 2. Summary Table
    print("\\n📋 EXPERIMENT SUMMARY TABLE")
    print("-" * 80)
    summary_df = pd.DataFrame(exp_comparison).T
    summary_df.index = [exp.replace('random_forgetting_', '').replace('_RL_tweak_conservative', '') 
                        for exp in summary_df.index]
    print(summary_df.round(4))
    
    # 3. Key Insights
    print("\\n💡 KEY INSIGHTS")
    print("-" * 30)
    
    # Find most affected experiment
    most_affected_exp = max(exp_comparison.keys(), key=lambda x: exp_comparison[x]['avg_change'])
    print(f"🔥 Most affected experiment: {most_affected_exp.replace('random_forgetting_', '').replace('_RL_tweak_conservative', '')}")
    print(f"   Average change: {exp_comparison[most_affected_exp]['avg_change']:.4f}")
    
    # Find most stable experiment  
    least_affected_exp = min(exp_comparison.keys(), key=lambda x: exp_comparison[x]['avg_change'])
    print(f"🛡️  Most stable experiment: {least_affected_exp.replace('random_forgetting_', '').replace('_RL_tweak_conservative', '')}")
    print(f"   Average change: {exp_comparison[least_affected_exp]['avg_change']:.4f}")
    
    # Layer type insights
    avg_by_type = {lt: np.mean(changes) for lt, changes in layer_type_changes.items()}
    most_vulnerable_type = max(avg_by_type.keys(), key=lambda x: avg_by_type[x])
    print(f"🎯 Most vulnerable layer type: {most_vulnerable_type}")
    print(f"   Average change: {avg_by_type[most_vulnerable_type]:.4f}")
    
    return exp_comparison, summary_df

# Run interactive analysis
exp_comparison, summary_df = create_interactive_analysis_dashboard()

## 8. Generate Custom Lucent Targets

Create custom targeting commands for specific analysis needs.

In [ ]:
# CUSTOM LUCENT TARGETING GENERATOR
print("🎯 CUSTOM LUCENT TARGETING")
print("="*50)

# Based on your actual analysis results, here are ready-to-use Lucent commands:

def generate_copy_paste_lucent_commands(lucent_targets, model_name="model"):
    """Generate copy-paste ready Lucent commands"""
    
    commands = []
    
    print("📋 COPY-PASTE READY LUCENT COMMANDS")
    print("-" * 50)
    print("Copy these commands directly into new Colab cells:")
    print()
    
    for i, target_info in enumerate(lucent_targets[:10]):
        target = target_info['target']
        change_mag = target_info['change_magnitude']
        
        # Generate the command
        command = f'''# Visualization {i+1}: {target} (Change: {change_mag:.4f})
img_{i+1} = render.render_vis({model_name}, "{target}", show_inline=True, thresholds=(512,))'''
        
        commands.append(command)
        print(command)
        print()
    
    # Also generate a batch command
    batch_command = f'''
# BATCH VISUALIZATION - Run all top targets at once
targets = {[t['target'] for t in lucent_targets[:6]]}

import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, target in enumerate(targets):
    img = render.render_vis({model_name}, target, show_inline=False, thresholds=(256,))
    if hasattr(img, 'cpu'):
        img_np = img.cpu().numpy().transpose(1, 2, 0)
    else:
        img_np = np.array(img)
    axes[i].imshow(img_np)
    axes[i].set_title(target, fontsize=10)
    axes[i].axis('off')

plt.suptitle('Top 6 Most Affected Components', fontsize=16)
plt.tight_layout()
plt.show()
'''
    
    print("🔥 BATCH VISUALIZATION COMMAND:")
    print("-" * 40)
    print(batch_command)
    
    return commands

# Generate commands
commands = generate_copy_paste_lucent_commands(lucent_targets)

# Generate targeting for different analysis focuses
print("\\n🎯 SPECIALIZED TARGETING OPTIONS")
print("-" * 50)

print("🔍 For Early Layer Analysis:")
early_targets = [t for t in lucent_targets if 'layer1' in t['target'] or 'layer2' in t['target']][:3]
for target in early_targets:
    print(f'render.render_vis(model, "{target["target"]}", show_inline=True)')

print("\\n🔍 For Deep Layer Analysis:")
deep_targets = [t for t in lucent_targets if 'layer3' in t['target'] or 'layer4' in t['target']][:3]
for target in deep_targets:
    print(f'render.render_vis(model, "{target["target"]}", show_inline=True)')

print("\\n🔍 For Channel-Specific Analysis:")
channel_targets = [t for t in lucent_targets if ':' in t['target']][:3]
for target in channel_targets:
    print(f'render.render_vis(model, "{target["target"]}", show_inline=True)')

print("\\n🎉 Your Lucent analysis setup is complete!")
print("💡 Use these commands in separate Colab cells for best results.")

## 🚀 Ready-to-Use Colab Commands

Here are the exact commands you can copy-paste into Google Colab cells to analyze your most influenced components!